## 1. Installation et imports

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes pandas numpy tqdm

In [ ]:
# À exécuter une seule fois (Colab / environnement vierge). Adapter les versions si besoin.
# !pip install -q -U transformers accelerate bitsandbytes pandas numpy tqdm

In [ ]:
!pip install --upgrade scipy scikit-learn numba transformers

In [ ]:
%pip install --force-reinstall --no-cache-dir numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 73.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.4
    Uninstalling numpy-2.5.4:
      Successfully uninstalled numpy-2.5.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.0 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.68.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.68.0 which is incompatible.


In [ ]:
import os, re, json, math, time, random, hashlib, unicodedata, textwrap, platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed

pd.set_option("display.max_colwidth", 200)
print("python", platform.python_version(), "| torch", torch.__version__, "| transformers", transformers.__version__)

python 3.13.15 | torch 2.11.0+cu130 | transformers 5.19.0


## 2. Configuration

Paramètres importants (tous modifiables ici) :
- **Décodage glouton** (`do_sample=False`) : réponses reproductibles, indispensables pour comparer A/B/C. Les trois configurations devront utiliser *les mêmes* paramètres de génération.
- `MAX_NEW_TOKENS` : coupe les réponses trop longues. Une réponse tronquée peut perdre sa ligne « Source ».
- `BATCH_SIZE` : à baisser en cas de `CUDA out of memory`.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
USE_4BIT = True            # Quantification NF4. Gardée même si la VRAM permettrait mieux : voir note en section 4.
SEED = 42
MAX_NEW_TOKENS = 300       # réponses du modèle évalué
JUDGE_MAX_NEW_TOKENS = 40  # sortie JSON du juge
BATCH_SIZE = 4

TEST_SET_PATH = Path("test_set_b1.jsonl")      # à placer à côté du notebook
OUTPUT_DIR = Path("outputs/step1_baseline")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ABSTAIN_MARKER = "INFORMATION ABSENTE"        # chaîne d'abstention imposée dans le prompt (réutilisée en B et C)

random.seed(SEED); np.random.seed(SEED); set_seed(SEED)

## 3. Vérification du matériel



In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "Aucun GPU CUDA détecté. Ce notebook utilise transformers + bitsandbytes (NVIDIA uniquement). "
        "Alternatives : Colab/Kaggle avec GPU, ou une chaîne llama.cpp (GGUF) / MLX à mettre en place séparément."
    )

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
bf16_ok = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if bf16_ok else torch.float16   # T4 : pas de bf16 -> float16

print(f"GPU : {props.name} | VRAM : {vram_gb:.1f} Go | bf16 : {bf16_ok} | dtype de calcul : {COMPUTE_DTYPE}")
if vram_gb < 7:
    print("ATTENTION : moins de 7 Go de VRAM, le chargement 4-bit risque d'échouer ou de déborder sur le CPU.")

GPU : Tesla T4 | VRAM : 15.6 Go | bf16 : True | dtype de calcul : torch.bfloat16


## 4. Chargement du modèle (4-bit NF4)


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
) if USE_4BIT else None

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")  # left padding : requis pour la génération par lots

t0 = time.time()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
)
model.eval()
print(f"Chargé en {time.time()-t0:.0f}s | empreinte mémoire : {model.get_memory_footprint()/1e9:.1f} Go")

offloaded = {k: v for k, v in getattr(model, "hf_device_map", {}).items() if str(v) in ("cpu", "disk")}
if offloaded:
    print("ATTENTION : des couches sont déportées sur CPU/disque (très lent) :", list(offloaded)[:5], "...")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Chargé en 251s | empreinte mémoire : 5.4 Go


## 5. Fonction de génération

Génération gloutonne par lots.

In [ ]:
@torch.inference_mode()
def generate_batch(list_of_messages, max_new_tokens=MAX_NEW_TOKENS):
    """list_of_messages : liste de conversations [{"role":..., "content":...}, ...]. Retourne une liste de textes."""
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in list_of_messages]
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,                       # glouton
        temperature=None, top_p=None, top_k=None,  # neutralise les valeurs par défaut de generation_config
        pad_token_id=tokenizer.pad_token_id,
    )
    gen = out[:, enc["input_ids"].shape[1]:]
    return [tokenizer.decode(g, skip_special_tokens=True).strip() for g in gen]

# Test de fumée
print(generate_batch([[{"role": "user", "content": "Réponds uniquement par : OK"}]], max_new_tokens=10))

['OK']


## 6. Jeu de test

Le fichier contient des questions **dans** le corpus (avec réponse de référence, fiches/pages de référence et faits-clés) et des questions **hors** corpus.


In [ ]:
def load_test_set(path):
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    ids = [r["id"] for r in rows]
    assert len(ids) == len(set(ids)), "identifiants dupliqués"
    valid_cat = {"in_corpus", "out_corpus_near", "out_corpus_offtopic"}
    for r in rows:
        assert r["category"] in valid_cat, r["id"]
        if r["category"] == "in_corpus":
            assert r["gold_answer"] and r["key_facts"] and r["gold_fiches"] and r["gold_pages"], r["id"]
    return pd.DataFrame(rows)

df_test = load_test_set(TEST_SET_PATH)
TEST_SET_SHA256 = hashlib.sha256(TEST_SET_PATH.read_bytes()).hexdigest()
print("SHA-256 du jeu de test :", TEST_SET_SHA256[:16], "...")
display(df_test.category.value_counts().to_frame("n"))
display(df_test[["id", "category", "question"]].head(8))

SHA-256 du jeu de test : f9ce59c626039960 ...


,n
category,
in_corpus,33
out_corpus_near,13
out_corpus_offtopic,4


,id,category,question
0,in01,in_corpus,Dans quel délai faut-il notifier une violation de données personnelles à la CNIL ?
1,in02,in_corpus,Quelle entropie minimale est exigée par défaut pour un mot de passe ?
2,in03,in_corpus,Quelle entropie minimale est acceptée pour un mot de passe lorsque des mesures complémentaires de restriction d'accès sont en place ?
3,in04,in_corpus,"Quelle entropie minimale est acceptée pour un code à 4 chiffres protégeant un matériel détenu par l'utilisateur, comme une carte SIM ?"
4,in05,in_corpus,Quelles fonctions de hachage sont jugées obsolètes pour stocker l'empreinte des mots de passe ?
5,in06,in_corpus,Quels algorithmes le guide cite-t-il pour stocker les mots de passe ?
6,in07,in_corpus,Pendant combien de temps faut-il conserver les journaux d'événements (traces) ?
7,in08,in_corpus,En quoi consiste la règle de sauvegarde « 3-2-1 » ?


## 7. Prompts de la baseline



In [ ]:
SYSTEM_BASE = (
    "Tu es un assistant réglementaire spécialisé dans la sécurité des données personnelles. "
    "Tu réponds en français, de façon concise."
)
SYSTEM_ABSTENTION = (
    SYSTEM_BASE + " "
    "Tu dois répondre UNIQUEMENT à partir du « Guide pratique RGPD - Sécurité des données personnelles » "
    "de la CNIL (version 2024, mise à jour 2026). Tu n'as PAS ce document sous les yeux. "
    f"Si tu n'es pas certain que l'information figure dans ce guide, réponds exactement : {ABSTAIN_MARKER}. "
    "Sinon, réponds en 3 phrases maximum, puis termine par une ligne de la forme "
    "« Source : Fiche n°X, page Y »."
)
PROMPT_VARIANTS = {"naive": SYSTEM_BASE, "abstention_aware": SYSTEM_ABSTENTION}

def build_messages(variant, question):
    return [{"role": "system", "content": PROMPT_VARIANTS[variant]},
            {"role": "user", "content": question}]

## 8. Génération des réponses

In [ ]:
def run_generation(variant, df, out_path):
    done = {}
    if out_path.exists():
        for l in open(out_path, encoding="utf-8"):
            r = json.loads(l); done[r["id"]] = r
    todo = [r for r in df.to_dict("records") if r["id"] not in done]
    with open(out_path, "a", encoding="utf-8") as f:
        for i in tqdm(range(0, len(todo), BATCH_SIZE), desc=f"génération [{variant}]"):
            batch = todo[i:i + BATCH_SIZE]
            outs = generate_batch([build_messages(variant, r["question"]) for r in batch])
            for r, o in zip(batch, outs):
                f.write(json.dumps({"id": r["id"], "variant": variant, "answer": o}, ensure_ascii=False) + "\n")
            f.flush()
    return pd.DataFrame([json.loads(l) for l in open(out_path, encoding="utf-8")])

generations = {}
for variant in PROMPT_VARIANTS:
    generations[variant] = run_generation(variant, df_test, OUTPUT_DIR / f"raw_{variant}.jsonl")
    print(variant, "->", len(generations[variant]), "réponses")

génération [naive]:   0%|          | 0/13 [00:00<?, ?it/s]

naive -> 50 réponses


génération [abstention_aware]:   0%|          | 0/13 [00:00<?, ?it/s]

abstention_aware -> 50 réponses


## 9. Évaluation

### 9.1 Règles déterministes : faits-clés et citations

In [ ]:
def norm(s):
    """minuscules, sans accents, espaces normalisés (y compris insécables)."""
    s = unicodedata.normalize("NFKD", s.lower())
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[\s\u00a0\u202f]+", " ", s).strip()

def fact_recall(answer, key_facts):
    a = norm(answer)
    hits = [any(re.search(norm(p), a) for p in alts) for alts in key_facts]
    return sum(hits) / len(hits)

FICHE_RE = re.compile(r"fiches?\s*(?:n\s*°|n°|no\.?|numero)?\s*((?:\d{1,2}(?:\s*(?:,|et|&|ou|/)\s*)?)+)")
PAGE_RE = re.compile(r"\b(?:pages?|p\.)\s*(?:n\s*°)?\s*(\d{1,3})")

def parse_fiches(answer):
    found = set()
    for m in FICHE_RE.finditer(norm(answer)):
        found |= {int(x) for x in re.findall(r"\d{1,2}", m.group(1))}
    return sorted(found)

def parse_pages(answer):
    return sorted({int(x) for x in PAGE_RE.findall(norm(answer))})

# Vérifications rapides du parseur
assert parse_fiches("Source : Fiche n°4, page 12") == [4]
assert parse_fiches("Source : fiches 4 et 21") == [4, 21]
assert parse_pages("Source : Fiche n°4, page 12") == [12]
assert fact_recall("Le délai est de 72 heures.", [["72 ?h", "72 heures"]]) == 1.0
print("parseurs OK")

parseurs OK


### 9.2 Juge LLM local

Le juge est **Qwen lui-même** (auto-évaluation, mêmes angles morts) 

In [ ]:
VALID_IN = {"CORRECT", "PARTIAL", "INCORRECT", "ABSTAIN"}
VALID_OOC = {"ABSTAIN", "HEDGED", "ASSERTED"}
JUDGE_SYSTEM = "You are a strict, impartial evaluator. Reply with one JSON object and nothing else."

def judge_messages(row):
    if row["category"] == "in_corpus":
        user = f"""Evaluate an answer about a French regulatory guide (CNIL).
Question: {row['question']}
Reference answer (ground truth from the guide): {row['gold_answer']}
Model answer: {row['answer']}

Return JSON: {{"verdict": "CORRECT|PARTIAL|INCORRECT|ABSTAIN", "relevant": 0 or 1}}
- CORRECT: contains the key information of the reference, no contradiction.
- PARTIAL: contains only part of the key information, or misses an essential element.
- INCORRECT: contradicts the reference or gives different information.
- ABSTAIN: the model declines to answer or says the information is unavailable.
"relevant" = 1 if the answer addresses the question asked (even if wrong), 0 if off-topic or an abstention."""
    else:
        user = f"""The reference guide does NOT contain the answer to the question below.
Question: {row['question']}
Model answer: {row['answer']}

Classify the model answer. Return JSON: {{"verdict": "ABSTAIN|HEDGED|ASSERTED"}}
- ABSTAIN: explicitly declines or says the information is not available, with no substantive content.
- HEDGED: says it is not in the guide or it is unsure, BUT still provides substantive content from general knowledge.
- ASSERTED: states a substantive answer as fact, or attributes it to the guide."""
    return [{"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": user}]

def is_explicit_abstention(answer):
    """Marqueur d'abstention suivi de (presque) rien : décision déterministe, sans juge."""
    a = answer.upper()
    if ABSTAIN_MARKER not in a:
        return False
    rest = re.sub(r"SOURCE\s*:.*", "", a.replace(ABSTAIN_MARKER, ""), flags=re.S)
    return len(re.findall(r"\w+", rest)) <= 12

def parse_judge(text, in_corpus):
    m = re.search(r"\{.*?\}", text, flags=re.S)
    try:
        obj = json.loads(m.group(0))
        v = str(obj.get("verdict", "")).upper()
        if v not in (VALID_IN if in_corpus else VALID_OOC):
            return "PARSE_ERROR", np.nan
        return v, (float(obj.get("relevant", 0)) if in_corpus else np.nan)
    except Exception:
        return "PARSE_ERROR", np.nan

def judge_dataframe(d):
    d = d.reset_index(drop=True).copy()
    verdicts, relevant, todo = [None] * len(d), [np.nan] * len(d), []
    for i, r in d.iterrows():
        if is_explicit_abstention(r["answer"]):
            verdicts[i] = "ABSTAIN"; relevant[i] = 0.0 if r["category"] == "in_corpus" else np.nan
        else:
            todo.append(i)
    step = BATCH_SIZE * 2
    for s in tqdm(range(0, len(todo), step), desc="juge"):
        idx = todo[s:s + step]
        outs = generate_batch([judge_messages(d.loc[i]) for i in idx], max_new_tokens=JUDGE_MAX_NEW_TOKENS)
        for i, o in zip(idx, outs):
            verdicts[i], relevant[i] = parse_judge(o, d.loc[i, "category"] == "in_corpus")
    d["verdict"], d["relevant"] = verdicts, relevant
    return d

### 9.3 Assemblage par variante

In [ ]:
def evaluate_variant(variant):
    d = df_test.merge(generations[variant][["id", "answer"]], on="id", how="left")
    assert d.answer.notna().all(), "réponses manquantes"
    d["variant"] = variant
    d["cited_fiches"] = d.answer.apply(parse_fiches)
    d["cited_pages"] = d.answer.apply(parse_pages)
    d["fact_recall"] = [fact_recall(a, kf) if c == "in_corpus" else np.nan
                        for a, kf, c in zip(d.answer, d.key_facts, d.category)]
    d["source_hit"] = [bool(set(cf) & set(gf)) for cf, gf in zip(d.cited_fiches, d.gold_fiches)]
    d["page_hit"] = [bool(set(cp) & set(gp)) for cp, gp in zip(d.cited_pages, d.gold_pages)]
    d["cit_precision"] = [len(set(cf) & set(gf)) / len(set(cf)) if cf else np.nan
                          for cf, gf in zip(d.cited_fiches, d.gold_fiches)]
    return judge_dataframe(d)

evals = {v: evaluate_variant(v) for v in PROMPT_VARIANTS}
for v, d in evals.items():
    print(v, "| erreurs de parsing du juge :", int((d.verdict == "PARSE_ERROR").sum()))

juge:   0%|          | 0/7 [00:00<?, ?it/s]

juge:   0%|          | 0/5 [00:00<?, ?it/s]

naive | erreurs de parsing du juge : 0
abstention_aware | erreurs de parsing du juge : 1


## 10. Métriques

In [ ]:
def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n; den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return centre - half, centre + half

def rate(mask):
    mask = pd.Series(mask).astype(bool); k, n = int(mask.sum()), len(mask)
    if n == 0:
        return "n/a"
    lo, hi = wilson(k, n)
    return f"{k/n:.0%} [{lo:.0%}-{hi:.0%}] ({k}/{n})"

def compute_metrics(d):
    ind = d[d.category == "in_corpus"]
    ooc = d[d.category != "in_corpus"]
    near = d[d.category == "out_corpus_near"]
    off = d[d.category == "out_corpus_offtopic"]
    citing = ind[ind.cited_fiches.apply(len) > 0]
    m = {
        "IN | fact_recall (moyenne)": f"{ind.fact_recall.mean():.0%}",
        "IN | exactitude stricte (CORRECT)": rate(ind.verdict == "CORRECT"),
        "IN | exactitude large (CORRECT+PARTIAL)": rate(ind.verdict.isin(["CORRECT", "PARTIAL"])),
        "IN | fausse abstention": rate(ind.verdict == "ABSTAIN"),
        "IN | pertinence": rate(ind.relevant == 1),
        "IN | citation présente": rate(ind.cited_fiches.apply(len) > 0),
        "IN | source pertinente (fiche)": rate(ind.source_hit),
        "IN | précision des citations (parmi citants)": f"{citing.cit_precision.mean():.0%} (n={len(citing)})" if len(citing) else "n/a",
        "IN | page correcte": rate(ind.page_hit),
        "IN | retrieval recall@k / precision@k": "N/A (pas de retrieval en A)",
        "OOC | abstention correcte (tous)": rate(ooc.verdict == "ABSTAIN"),
        "OOC | abstention correcte (proches)": rate(near.verdict == "ABSTAIN"),
        "OOC | abstention correcte (hors sujet)": rate(off.verdict == "ABSTAIN"),
        "OOC | réponse injustifiée (HEDGED+ASSERTED)": rate(ooc.verdict.isin(["HEDGED", "ASSERTED"])),
        "OOC | hallucination (ASSERTED)": rate(ooc.verdict == "ASSERTED"),
        "OOC | citation parasite": rate(ooc.cited_fiches.apply(len) > 0),
        "ALL | erreurs de parsing du juge": f"{int((d.verdict == 'PARSE_ERROR').sum())}/{len(d)}",
    }
    return pd.Series(m)

metrics_table = pd.DataFrame({v: compute_metrics(d) for v, d in evals.items()})
display(metrics_table)

,naive,abstention_aware
IN | fact_recall (moyenne),58%,39%
IN | exactitude stricte (CORRECT),55% [38%-70%] (18/33),39% [25%-56%] (13/33)
IN | exactitude large (CORRECT+PARTIAL),88% [73%-95%] (29/33),70% [53%-83%] (23/33)
IN | fausse abstention,0% [0%-10%] (0/33),15% [7%-31%] (5/33)
IN | pertinence,100% [90%-100%] (33/33),85% [69%-93%] (28/33)
IN | citation présente,0% [0%-10%] (0/33),94% [80%-98%] (31/33)
IN | source pertinente (fiche),0% [0%-10%] (0/33),6% [2%-20%] (2/33)
IN | précision des citations (parmi citants),n/a,6% (n=31)
IN | page correcte,0% [0%-10%] (0/33),3% [1%-15%] (1/33)
IN | retrieval recall@k / precision@k,N/A (pas de retrieval en A),N/A (pas de retrieval en A)


## 11. Analyse d'erreurs et export

In [ ]:
def show_cases(d, mask, cols=("id", "question", "answer", "verdict"), n=6, width=110):
    sub = d[mask].head(n)
    print(f"--- {len(d[mask])} cas (affichage de {len(sub)}) ---")
    for _, r in sub.iterrows():
        print(f"[{r['id']}] Q : {r['question']}")
        if r["category"] == "in_corpus":
            print(f"    Réf : {r['gold_answer']}  (fiches {r['gold_fiches']}, pages {r['gold_pages']})")
        print("    Rép :", textwrap.shorten(r["answer"].replace("\n", " "), width=400))
        print(f"    -> verdict={r['verdict']} | fiches citées={r['cited_fiches']} | pages citées={r['cited_pages']}\n")

A = evals["abstention_aware"]
print("=== A (abstention_aware) : hallucinations hors corpus ===")
show_cases(A, A.verdict == "ASSERTED")

=== A (abstention_aware) : hallucinations hors corpus ===
--- 5 cas (affichage de 5) ---
[oc01] Q : Quel est le montant maximal de l'amende encourue en cas de violation du RGPD ?
    Rép : Le montant maximal de l'amende encourue peut atteindre 20 millions d'euros ou 4% du chiffre d'affaires annuel mondial, selon lequel est plus élevé. Source : Fiche n°1, page 5.
    -> verdict=ASSERTED | fiches citées=[1] | pages citées=[5]

[oc04] Q : Quels paramètres (facteur de coût, nombre d'itérations) le guide recommande-t-il pour bcrypt ?
    Rép : Le guide recommande d'utiliser un facteur de coût entre 12 et 14 pour bcrypt, et un nombre d'itérations minimal de 100 000. Source : Fiche n°5, page 18.
    -> verdict=ASSERTED | fiches citées=[5] | pages citées=[18]

[oc06] Q : Quelle est la durée de validité maximale recommandée pour un certificat TLS ?
    Rép : La durée de validité maximale recommandée pour un certificat TLS est de 2 ans. Source : Fiche n°12, page 37.
    -> verdict=ASSERTED | fic

In [ ]:
print("=== A (abstention_aware) : réponses INCORRECTES en corpus ===")
show_cases(A, (A.category == "in_corpus") & (A.verdict == "INCORRECT"))
print("=== A (abstention_aware) : fausses abstentions en corpus ===")
show_cases(A, (A.category == "in_corpus") & (A.verdict == "ABSTAIN"), n=4)
print("=== A (abstention_aware) : citations fausses (fiche citée mais aucune bonne) ===")
show_cases(A, (A.category == "in_corpus") & (A.cited_fiches.apply(len) > 0) & (~A.source_hit), n=4)

=== A (abstention_aware) : réponses INCORRECTES en corpus ===
--- 5 cas (affichage de 5) ---
[in01] Q : Dans quel délai faut-il notifier une violation de données personnelles à la CNIL ?
    Réf : Dans les 72 heures (notification possible en deux temps : une initiale dans les 72 h, puis une complémentaire si nécessaire).  (fiches [19], pages [42])
    Rép : Il est nécessaire de notifier la CNIL dans les trente jours suivant la détection de la violation de données personnelles. Source : Fiche n°1, page 15.
    -> verdict=INCORRECT | fiches citées=[1] | pages citées=[15]

[in09] Q : À quelle fréquence minimale faut-il revoir les habilitations des utilisateurs ?
    Réf : Au moins une fois par an (revue au moins annuelle).  (fiches [5], pages [14])
    Rép : La fréquence minimale de révision des habilitations des utilisateurs n'est pas fixée par le RGPD. Source : Fiche n°15, page 37.
    -> verdict=INCORRECT | fiches citées=[15] | pages citées=[37]

[in23] Q : Quels logiciels de scan de v

In [ ]:
# Sauvegarde : résultats complets, métriques, métadonnées de reproductibilité
for v, d in evals.items():
    d.to_csv(OUTPUT_DIR / f"results_{v}.csv", index=False)
metrics_table.to_csv(OUTPUT_DIR / "metrics_baseline.csv")

meta = {
    "step": "1_baseline", "model": MODEL_ID, "quant_4bit": USE_4BIT, "compute_dtype": str(COMPUTE_DTYPE),
    "seed": SEED, "max_new_tokens": MAX_NEW_TOKENS, "batch_size": BATCH_SIZE, "decoding": "greedy",
    "test_set_sha256": TEST_SET_SHA256, "n_questions": int(len(df_test)),
    "gpu": props.name, "transformers": transformers.__version__, "torch": torch.__version__,
    "reference_config_A_variant": "abstention_aware",
    "date": time.strftime("%Y-%m-%d %H:%M:%S"),
}
json.dump(meta, open(OUTPUT_DIR / "meta.json", "w"), indent=2)

# Échantillon à annoter à la main pour mesurer l'accord juge / humain (colonne human_verdict à remplir)
sample = A.sample(n=min(20, len(A)), random_state=SEED)[["id", "category", "question", "gold_answer", "answer", "verdict"]].copy()
sample = sample.rename(columns={"verdict": "judge_verdict"}); sample["human_verdict"] = ""
sample.to_csv(OUTPUT_DIR / "to_annotate_A.csv", index=False)
print("Fichiers écrits dans", OUTPUT_DIR.resolve())

Fichiers écrits dans /content/outputs/step1_baseline


In [ ]:
# À exécuter APRÈS avoir rempli la colonne human_verdict de to_annotate_A.csv
# (valeurs : CORRECT/PARTIAL/INCORRECT/ABSTAIN en corpus ; ABSTAIN/HEDGED/ASSERTED hors corpus)
ann = pd.read_csv(OUTPUT_DIR / "to_annotate_A_filled.csv").dropna(subset=["human_verdict"])
ann = ann[ann.human_verdict.astype(str).str.strip() != ""]
if len(ann):
    agree = (ann.human_verdict.str.upper().str.strip() == ann.judge_verdict).mean()
    print(f"Accord juge/humain : {agree:.0%} sur {len(ann)} exemples")
else:
    print("Aucune annotation manuelle trouvée pour l'instant.")

Accord juge/humain : 80% sur 20 exemples
